### Construção do dataset final

Monta o dataset a partir dos intermediários (histórico). **Para o dataset usado na ML**, rode depois `03_realign_features_by_electrode.ipynb` (features por nome de eletrodo).

Fluxo:
1. Lê features intermediárias (`data/intermediate/sub-*.parquet`)
2. Faz merge com metadados (`participants.csv`)
3. Aplica filtro de amostra saudável
4. Gera dummies de metadados
5. Salva `data/processed/dataset_features.parquet`

Pré-requisito: `01_build_networks.ipynb`. Ordem recomendada: `01` → `03_realign` → EDA/ML (este `02` documenta o merge/filtro; o `03` regenera o parquet final alinhado).


### Imports


In [1]:
from pathlib import Path

import pandas as pd


### Paths


In [2]:
INTERMEDIATE_DIR = Path("../../data/intermediate")
METADATA_FILE = Path("../../data/metadata/participants.csv")
OUTPUT_DIR = Path("../../data/processed")
OUTPUT_FILE = OUTPUT_DIR / "dataset_features.parquet"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("INTERMEDIATE_DIR:", INTERMEDIATE_DIR.resolve())
print("METADATA_FILE:", METADATA_FILE.resolve())
print("OUTPUT_FILE:", OUTPUT_FILE.resolve())


INTERMEDIATE_DIR: C:\Users\Anne\OneDrive\Documentos\Google Drive\Ciência de Dados\MBA USP\TCC\Código\TCC\data\intermediate
METADATA_FILE: C:\Users\Anne\OneDrive\Documentos\Google Drive\Ciência de Dados\MBA USP\TCC\Código\TCC\data\metadata\participants.csv
OUTPUT_FILE: C:\Users\Anne\OneDrive\Documentos\Google Drive\Ciência de Dados\MBA USP\TCC\Código\TCC\data\processed\dataset_features.parquet


### Carregar features intermediárias


In [3]:
print("Lendo features intermediárias...")
files = sorted(INTERMEDIATE_DIR.glob("sub-*.parquet"))
print(f"Encontrados {len(files)} arquivos")

dfs = []
for file in files:
    subject_id, condition = file.stem.split("_", 1)
    df = pd.read_parquet(file)
    df["subject_id"] = subject_id
    df["condition"] = condition
    dfs.append(df)

df_features = pd.concat(dfs, ignore_index=True)
df_features.shape


Lendo features intermediárias...
Encontrados 236 arquivos


(236, 374)

### Merge com metadados


In [4]:
print("Lendo metadados...")
meta = pd.read_csv(METADATA_FILE)
meta["subject_id"] = meta["ID"]

df_final = df_features.merge(
    meta,
    on="subject_id",
    how="left",
    validate="m:1",
)

Lendo metadados...


In [5]:
df_final

,degree_0,degree_1,degree_2,degree_3,degree_4,degree_5,degree_6,degree_7,degree_8,degree_9,...,SKID_Diagnoses 1,SKID_Diagnoses 2,Comments_SKID_assessment,Hamilton_Scale,BSL23_sumscore,BSL23_behavior,AUDIT,Standard_Alcoholunits_Last_28days,Alcohol_Dependence_In_1st-3rd_Degree_relative,Relationship_Status
0,0.446221,0.424809,0.544476,0.432554,0.399805,0.430520,0.564116,0.578766,0.524267,0.546721,...,"#CODE:296.26 #DESC:Major Depressive Disorder,...",NaN,past subclinical dperession,0.0,7.0,1.0,7.0,29.5,No,No
1,0.537599,0.530284,0.562295,0.521064,0.528043,0.534308,0.553488,0.568283,0.483891,0.508328,...,#CODE:V71.09 #DESC:No Diagnosis or Condition ...,NaN,NaN,4.0,6.0,2.0,11.0,53.5,Yes (9),No
2,0.524312,0.467778,0.546044,0.475507,0.476817,0.464719,0.509629,0.543502,0.405685,0.444021,...,#CODE:V71.09 #DESC:No Diagnosis or Condition ...,NaN,NaN,4.0,6.0,2.0,11.0,53.5,Yes (9),No
3,0.617032,0.580945,0.619511,0.546339,0.531890,0.560235,0.625229,0.592469,0.561956,0.638411,...,#CODE:V71.09 #DESC:No Diagnosis or Condition ...,NaN,NaN,4.0,3.0,1.0,3.0,5,No,Yes
4,0.553264,0.551601,0.539167,0.506458,0.472431,0.505821,0.609756,0.563717,0.537651,0.621613,...,#CODE:V71.09 #DESC:No Diagnosis or Condition ...,NaN,NaN,4.0,3.0,1.0,3.0,5,No,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
231,0.436077,0.402551,0.412220,0.465841,0.416453,0.374553,0.285576,0.446587,0.428470,0.415520,...,#CODE:V71.09 #DESC:No Diagnosis or Condition ...,NaN,NaN,0.0,0.0,0.0,3.0,6,Yes,Yes
232,0.524657,0.388739,0.543432,0.528492,0.430382,0.508736,0.477252,0.585333,0.447551,0.451690,...,#CODE:V71.09 #DESC:No Diagnosis or Condition ...,NaN,NaN,4.0,1.0,0.0,3.0,26,No,No
233,0.516514,0.455330,0.513625,0.451788,0.341061,0.395840,0.488951,0.560125,0.407389,0.368827,...,#CODE:V71.09 #DESC:No Diagnosis or Condition ...,NaN,NaN,4.0,1.0,0.0,3.0,26,No,No
234,0.613349,0.616589,0.674175,0.581415,0.486011,0.540270,0.645903,0.660474,0.448364,0.499186,...,#CODE:V71.09 #DESC:No Diagnosis or Condition ...,NaN,NaN,2.0,8.0,0.0,2.0,12,yes,yes


In [6]:

df_final = df_final.rename(
    columns={
        "Gender_ 1=female_2=male": "gender",
        "Age": "age",
        "Handedness": "handedness",
        "Education": "education",
        "DRUG_0=negative_1=Positive": "drug",
        "Smoking_num_(Non-smoker=1, Occasional Smoker=2, Smoker=3)": "smoking",
        "SKID_Diagnoses": "skid_diagnoses",
        "SKID_Diagnoses 1": "skid_diagnoses_1",
        "SKID_Diagnoses 2": "skid_diagnoses_2",
        "Comments_SKID_assessment": "comments_skid_assessment",
        "Hamilton_Scale": "hamilton_scale",
        "BSL23_sumscore": "bsl23_sumscore",
        "BSL23_behavior": "bsl23_behavior",
        "AUDIT": "audit",
        "Standard_Alcoholunits_Last_28days": "standard_alcoholunits_last_28days",
        "Alcohol_Dependence_In_1st-3rd_Degree_relative": "alcohol_dependence_in_1st_3rd_degree_relative",
        "Relationship_Status": "relationship_status",
    }
)

print(f"Shape inicial: {df_final.shape}")
df_final[["subject_id", "condition", "age", "gender"]].head()


Shape inicial: (236, 395)


,subject_id,condition,age,gender
0,sub-010060,EO,30-35,2
1,sub-010061,EC,25-30,2
2,sub-010061,EO,25-30,2
3,sub-010062,EC,25-30,1
4,sub-010062,EO,25-30,1


### Padronizar strings e filtrar amostra saudável


In [7]:
df_final["relationship_status"] = df_final["relationship_status"].str.lower()
df_final["handedness"] = df_final["handedness"].str.lower()
df_final["education"] = df_final["education"].str.lower()

healthy_mask = (
    (df_final["drug"] == 0)
    & (df_final["smoking"] == 1)
    & (df_final["audit"] < 8)
    & (df_final["hamilton_scale"] <= 7)
    & (df_final["bsl23_behavior"] == 0)
    & (df_final["skid_diagnoses"].str.lower() == "none")
    & (df_final["skid_diagnoses_2"].isna())
    & (df_final["comments_skid_assessment"].isna())
)

df_healthy = df_final.loc[healthy_mask].copy()

print(f"Shape após filtro saudável: {df_healthy.shape}")
print(f"N indivíduos antes do filtro: {df_final['subject_id'].nunique()}")
print(f"N indivíduos após filtro saudável: {df_healthy['subject_id'].nunique()}")


Shape após filtro saudável: (108, 395)
N indivíduos antes do filtro: 119
N indivíduos após filtro saudável: 54


### Dummies de metadados


In [8]:
META_COLS = [
    "age",
    "gender",
    "handedness",
    "education",
    "relationship_status",
]

df_meta = df_healthy[META_COLS].copy()
print(f"Shape após seleção de metadados: {df_meta.shape}")

df_meta = pd.get_dummies(df_meta, columns=META_COLS, drop_first=True)
df_meta.head()


Shape após seleção de metadados: (108, 5)


,age_25-30,age_30-35,age_55-60,age_60-65,age_65-70,age_70-75,age_75-80,gender_2,handedness_left,handedness_right,education_gymnasium,education_realschule,relationship_status_yes
17,False,False,False,False,False,False,False,True,False,True,True,False,False
18,False,False,False,False,False,False,False,True,False,True,True,False,False
19,False,False,False,False,False,False,False,True,False,True,False,True,True
20,False,False,False,False,False,False,False,True,False,True,False,True,True
33,True,False,False,False,False,False,False,False,True,False,True,False,False


### Concatenar, corrigir E0 → EO e salvar


In [9]:
df_analysis = pd.concat([df_healthy, df_meta], axis=1)
df_analysis["condition"] = df_analysis["condition"].replace({"E0": "EO"})

print(f"Shape após concatenação: {df_analysis.shape}")
print(df_analysis["condition"].value_counts())

df_analysis.to_parquet(OUTPUT_FILE, index=False)
print(f"Dataset final salvo em: {OUTPUT_FILE.resolve()}")
print(f"Shape final: {df_analysis.shape}")
df_analysis.head()


Shape após concatenação: (108, 408)
condition
EC    54
EO    54
Name: count, dtype: int64
Dataset final salvo em: C:\Users\Anne\OneDrive\Documentos\Google Drive\Ciência de Dados\MBA USP\TCC\Código\TCC\data\processed\dataset_features.parquet
Shape final: (108, 408)


,degree_0,degree_1,degree_2,degree_3,degree_4,degree_5,degree_6,degree_7,degree_8,degree_9,...,age_60-65,age_65-70,age_70-75,age_75-80,gender_2,handedness_left,handedness_right,education_gymnasium,education_realschule,relationship_status_yes
17,0.415971,0.419330,0.384460,0.365096,0.333948,0.415580,0.386103,0.298642,0.353575,0.400329,...,False,False,False,False,True,False,True,True,False,False
18,0.466049,0.451235,0.496187,0.444153,0.375958,0.372649,0.426404,0.382513,0.337222,0.538553,...,False,False,False,False,True,False,True,True,False,False
19,0.573540,0.568852,0.573712,0.539383,0.471784,0.529896,0.601553,0.569455,0.456852,0.519599,...,False,False,False,False,True,False,True,False,True,True
20,0.387807,0.378893,0.472657,0.431546,0.397882,0.457414,0.508355,0.506541,0.421590,0.482511,...,False,False,False,False,True,False,True,False,True,True
33,0.667041,0.691087,0.669646,0.646260,0.664580,0.647651,0.629501,0.562688,0.509352,0.477082,...,False,False,False,False,False,True,False,True,False,False


### Próximo passo

Com o dataset pronto, use os notebooks de EDA:

- `notebook/01_eda/01_eda_global.ipynb`
- `notebook/01_eda/02_eda_node_features.ipynb`
- `notebook/01_eda/03_analise_descriva_redes.ipynb`
